In [2]:
import subprocess
import re
import platform

def get_connected_devices():
    try:
        if platform.system() == "Linux":
            arp_command = "arp -a"
        elif platform.system() == "Windows":
            arp_command = "arp -a"
        else:
            print("Système d'exploitation non pris en charge.")
            return {}

        # Exécute la commande pour obtenir la liste des appareils connectés
        arp_output = subprocess.check_output(arp_command, shell=True, stderr=subprocess.STDOUT).decode("utf-8", errors="ignore")

        # Extrait les adresses IP et les adresses MAC de la sortie
        ip_mac_pattern = re.compile(r"(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s+([0-9A-Fa-f]{2}[:-]){5}([0-9A-Fa-f]{2})")
        matches = ip_mac_pattern.findall(arp_output)

        # Crée un dictionnaire pour stocker les résultats (adresse IP -> adresse MAC)
        connected_devices = {}

        for match in matches:
            ip_address, mac_address = match[0], match[2].replace("-", ":") # Formatage de l'adresse MAC
            connected_devices[ip_address] = mac_address

        return connected_devices    
    except subprocess.CalledProcessError as e:
        print(f"Erreur lors de l'exécution de la commande : {e.output}")
        return {}

def get_rssi(ip_address):
    try:
        # Exécute la commande "ping" pour obtenir la force du signal (RSSI) de l'appareil
        ping_output = subprocess.check_output(["ping", "-c", "1", ip_address]).decode("utf-8")
        
        # Extrait la valeur RSSI de la sortie
        rssi_pattern = re.compile(r"(\d+) ms")
        rssi_match = rssi_pattern.search(ping_output)

        if rssi_match:
            rssi = int(rssi_match.group(1))
            return rssi
        else:
            return None
    except subprocess.CalledProcessError as e:
        print(f"Erreur lors de l'exécution de la commande ping : {e.output}")
        return None

def main():
    connected_devices = get_connected_devices()

    print("Appareils connectés :")
    for ip_address, mac_address in connected_devices.items():
        rssi = get_rssi(ip_address)
        if rssi is not None:
            print(f"IP : {ip_address}, MAC : {mac_address}, RSSI : {rssi} ms")
        else:
            print(f"IP : {ip_address}, MAC : {mac_address}, RSSI : Non disponible")

if __name__ == "__main__":
    if platform.system() == "Linux" and os.getuid() != 0:
        print("Ce script nécessite des autorisations sudo pour exécuter la commande ping.")
        exit(1)
    main()


Appareils connectés :


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x88 in position 18: invalid start byte